# Prepare the training data -- once, on CPU

Downloads **Middlebury** (every release with public ground truth, both views' disparity) and
**KITTI 2012 + 2015** (the 394 labelled training pairs) from their official servers, adds
**InStereo2K** from a download you attach, checks every label by warping, and writes the result
to `/kaggle/working/stereo-data` (about 1.2 GB, plus about 1.5 GB with InStereo2K).

Make that output a dataset once, and the training notebook attaches it with **no download in the
GPU session**.

**Before running** (right-hand panel): Accelerator **None**, Internet **On**.
Then **Save Version -> Save & Run All**; it takes roughly 15-30 minutes, nearly all of it downloading.

### InStereo2K is a manual download

Its only hosts are OneDrive and Baidu, neither of which can be scripted, so fetch it once in a
browser from [the official page](https://github.com/YuhuaXu/StereoDataset), upload the file as a
**private** Kaggle dataset with *instereo* in its name, and attach it to this notebook. It is found
by that name; leave it unattached to skip InStereo2K.

In [ ]:
import os, subprocess, sys

GITHUB_USER, GITHUB_REPO, GITHUB_BRANCH = "hnquang-cs", "stereo-depth", "main"
INSTEREO2K = ""                # path to the attached download; "" finds it by name

# The code goes to /tmp, not /kaggle/working: everything in /kaggle/working becomes
# the notebook's output, and so would end up inside the dataset.
ROOT = f"/tmp/{GITHUB_REPO}"
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH,
                    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git", ROOT], check=True)
sys.path.insert(0, ROOT)

from stereo.data.prepare import find_instereo2k, prepare_all

INSTEREO2K = INSTEREO2K or find_instereo2k("/kaggle/input")
print(f"InStereo2K: {INSTEREO2K or 'not attached -- skipped'}")
prepare_all("/kaggle/working/stereo-data", cache="/tmp/stereo-prepare-cache",
            instereo2k=INSTEREO2K)

## Make it a dataset (once)

1. When the run has finished, open this notebook's **Output** tab: `stereo-data/` is there.
2. Choose **New Dataset** from that output, keep it **Private**, and name it, for example,
   `stereo-training-data`.
3. In the training notebook, **Add Input -> Your Datasets ->** that dataset. Nothing else to edit:
   it is found by its `stereo_data_manifest.json`, wherever Kaggle mounts it.

Keep it private: KITTI is licensed CC BY-NC-SA 3.0, InStereo2K is for non-commercial use, and
Middlebury asks to be cited.